## Step 4 — calculate building context using polygon-edge distances
**# of cells in notebook:** 1

**Purpose:** Calculate building neighborhood variables using polygon edge-to-edge distance. The building-level tessellation created in Step 3 is used to derive tessellation-based building measures such as `cell_area_m2` and `bldg_coverage_ratio`.

**Input:**

- the block-specific workspaces created in Step 2, each containing:
  - `block`
  - `buildings`
- `building_level_tessellation.gpkg` created in Step 3
- the citywide buildings layer with building area

**Output:**

Within each block folder:

- `building_context_features_new_dist.gpkg` — building features with polygon-edge-distance neighborhood variables and tessellation attributes
- `building_context_features_new_dist.csv` — tabular version of those attributes

At the base block directory:

- `building_context_features_new_dist_summary.csv` — summary information for all processed blocks

**Main logic:**

**Cell 1 — Calculate polygon-edge-distance building context**

1. Loads and cleans the citywide building polygons and builds a polygon spatial index.
2. Reads the buildings and block boundary for each selected block and preserves the FileGDB building feature ID as `context_bldg_id`.
3. Reads the building-level tessellation from Step 3 and calculates `cell_area_m2` from tessellation geometry.
4. Joins tessellation cell area to the corresponding source building and calculates `bldg_coverage_ratio` as building area divided by tessellation-cell area.
5. Finds nearby citywide building polygons and calculates exact polygon-to-polygon edge distances.
6. Calculates nearest-neighbor distance and neighbor-area measures, as well as building counts and area summaries within 30 m, 60 m, and 100 m edge-distance neighborhoods.
7. Writes the building-context outputs and an overall processing summary.


In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import box
from shapely.ops import unary_union

warnings.filterwarnings("ignore")

# =============================================================================
# BUILDING CONTEXT FEATURES - POLYGON EDGE DISTANCE VERSION
# =============================================================================
#
# Purpose
# -------
# Calculate building-context variables using polygon-to-polygon edge distances.
#
# This version DOES NOT recreate the building-level tessellation. It reads the
# existing building_level_tessellation.gpkg created in Step 3, calculates
# cell_area_m2 from tessellation geometry, and joins that value back to the
# corresponding source building using context_bldg_id. It then calculates
# bldg_coverage_ratio = building area / cell_area_m2.
#
# Neighbor-distance method
# ------------------------
# - Build a GeoPandas/Shapely spatial index over city-wide building polygons.
# - For each block building, find nearby candidate polygons.
# - Calculate exact polygon-to-polygon distance with Shapely geometry.distance().
# - Sort neighbors by edge distance.
#
# Important interpretation
# ------------------------
# For fixed-radius variables, a neighbor qualifies if its polygon edge distance
# from the focal building is <= the radius.
#
# Once a neighbor qualifies, the neighbor's ENTIRE building area is included in
# the area summary. The script does not calculate partial overlap with a buffer.
#
# Outputs
# -------
# In each block folder:
#   building_context_features_new_dist.gpkg
#   building_context_features_new_dist.csv
#
# At the base folder:
#   building_context_features_new_dist_summary.csv
#
# Existing tessellation file used as input:
#   building_level_tessellation.gpkg
#
# =============================================================================


# -----------------------------------------------------------------------------
# USER INPUTS
# -----------------------------------------------------------------------------

base_folder = r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"

city_buildings_gdb = r"E:\_johannesburg\_analysis\population\population.gdb"
city_buildings_layer = "johannesburg_buildings_utm35s"

area_field = "area_m_utm"

coverage_field = "bldg_coverage_ratio"

# Optional.
# If you have a stable unique building ID field present in BOTH the city-wide
# buildings layer and each block-level buildings layer, set it here.
#
# Example:
#   unique_building_id_field = "building_id"
#
# If left as None, the script removes the focal building from the neighbor list
# using exact geometry WKB matching. That is usually fine if block buildings are
# copied directly from the city-wide buildings layer.
unique_building_id_field = None

# K-nearest-neighbor variables.
# For each k in neighbor_ks, the script creates:
#   dist_nn{k}
#
# For each k in area_summary_ks, the script creates:
#   mean_area_nn{k}
#   sum_area_nn{k}
#   mean_log_area_nn{k}
neighbor_ks = [1, 5, 10, 20, 40]
area_summary_ks = [5, 10, 40]

# Fixed-radius variables, in meters.
# For each r, the script creates:
#   count_bldgs_{r}m
#   sum_area_{r}m
#   mean_area_{r}m
#   mean_log_area_{r}m
radius_values = [30, 60, 100]

# Adaptive search settings for edge-distance k-nearest neighbors.
#
# Because a spatial index does not directly provide exact polygon-edge-distance
# k-nearest neighbors, the script expands a search radius until it finds enough
# exact-distance candidates.
#
# For dense building data, initial_knn_search_distance=100 is usually enough.
# For sparse areas, the search expands up to max_knn_search_distance.
initial_knn_search_distance = 100
max_knn_search_distance = 5000
knn_expansion_factor = 2.0

overwrite_outputs = True

# Existing tessellation input inside each block folder.
existing_tess_gpkg_name = "building_level_tessellation.gpkg"
existing_tess_layer = "building_level_tessellation"

# Output names inside each block folder.
out_building_gpkg_name = "building_context_features_new_dist.gpkg"
out_building_csv_name = "building_context_features_new_dist.csv"
out_building_layer = "building_context_features_new_dist"


# -----------------------------------------------------------------------------
# HELPER FUNCTIONS
# -----------------------------------------------------------------------------

def safe_log_area(values):
    """
    Log-transform area values safely.
    Values <= 0 become NaN.
    """
    values = pd.to_numeric(values, errors="coerce").astype(float)
    out = np.full(len(values), np.nan, dtype=float)

    valid = np.isfinite(values) & (values > 0)
    out[valid] = np.log(values[valid])

    return out


def clean_geom(g):
    """
    Basic geometry cleaning.

    This does not snap to a precision grid.
    It only tries make_valid / buffer(0) to repair invalid polygons.
    """
    if g is None or g.is_empty:
        return None

    try:
        from shapely import make_valid
        g = make_valid(g)
    except Exception:
        try:
            g = g.buffer(0)
        except Exception:
            return None

    if g is None or g.is_empty:
        return None

    if not g.is_valid:
        try:
            g = g.buffer(0)
        except Exception:
            return None

    if g is None or g.is_empty:
        return None

    return g


def find_block_folders(base_folder):
    """
    Finds block folders that contain a FileGDB named after the folder.

    Example:
      E:\\...\\large_pop_blocks\\_397\\_397.gdb
    """
    block_folders = []

    for folder in sorted(glob.glob(os.path.join(base_folder, "_*"))):
        if not os.path.isdir(folder):
            continue

        folder_name = os.path.basename(folder)
        gdb_path = os.path.join(folder, f"{folder_name}.gdb")

        if os.path.exists(gdb_path):
            block_folders.append(folder)

    return block_folders


def read_gdb_layer(gdb_path, layer_name, preserve_fid=False):
    """Read a FileGDB layer with GeoPandas/pyogrio."""
    kwargs = {"engine": "pyogrio"}

    if preserve_fid:
        kwargs["fid_as_index"] = True

    return gpd.read_file(gdb_path, layer=layer_name, **kwargs)


def get_centroid_xy(gdf):
    """
    Returns centroid x/y arrays.

    This is retained only so centroid_x and centroid_y can be written as
    descriptive attributes. These centroids are NOT used for distance variables.
    """
    centroids = gdf.geometry.centroid
    x = centroids.x.to_numpy(dtype=float)
    y = centroids.y.to_numpy(dtype=float)
    return x, y


def expanded_bbox(geom, distance):
    """
    Creates a rectangular search envelope expanded by distance.

    The spatial index uses this envelope to find candidate geometries.
    Exact polygon-to-polygon distances are calculated afterward.
    """
    minx, miny, maxx, maxy = geom.bounds

    return box(
        minx - distance,
        miny - distance,
        maxx + distance,
        maxy + distance
    )


def sindex_query_positions(sindex, query_geom):
    """
    Compatibility wrapper for GeoPandas spatial index query results.

    Different GeoPandas/Shapely versions may return slightly different array
    types. This function standardizes the result to a list of integer positions.
    """
    result = sindex.query(query_geom)

    arr = np.asarray(result)

    if arr.size == 0:
        return []

    # Normal GeoPandas case: one-dimensional array of positions.
    if arr.ndim == 1:
        return arr.astype(int).tolist()

    # Defensive fallback in case a two-row pair array is returned.
    # For a single query geometry, the second row should contain candidate
    # positions.
    if arr.ndim == 2 and arr.shape[0] == 2:
        return arr[1].astype(int).tolist()

    return arr.ravel().astype(int).tolist()


def make_city_id_array(city_gdf, unique_building_id_field):
    """
    Returns a numpy array of city-wide building IDs if a usable ID field is set.
    """
    if unique_building_id_field is None:
        return None

    if unique_building_id_field not in city_gdf.columns:
        print(
            f"WARNING: unique_building_id_field='{unique_building_id_field}' "
            "was not found in the city-wide buildings layer. Falling back to WKB self-removal."
        )
        return None

    return city_gdf[unique_building_id_field].astype(str).to_numpy()


def get_focal_unique_id(row, unique_building_id_field):
    """
    Gets the focal building's unique ID if available.
    """
    if unique_building_id_field is None:
        return None

    if unique_building_id_field not in row.index:
        return None

    val = row[unique_building_id_field]

    if pd.isna(val):
        return None

    return str(val)


def get_edge_distance_candidates(
    focal_geom,
    city_gdf,
    city_sindex,
    search_distance,
    city_id_array=None,
    focal_unique_id=None,
    focal_wkb=None,
    remove_exact_self=True
):
    """
    Find candidate city-wide buildings within an expanded search box, then
    calculate exact polygon-to-polygon edge distances.

    Self-removal logic
    ------------------
    Preferred:
      If unique_building_id_field is configured and present in both datasets,
      remove candidates whose unique ID equals the focal unique ID.

    Fallback:
      Remove candidates whose geometry WKB exactly matches the focal geometry WKB.

    Important:
      The script does NOT remove all distance-zero candidates. Distinct buildings
      can touch or slightly overlap, and those should count as zero-distance
      neighbors.
    """

    if focal_geom is None or focal_geom.is_empty:
        return pd.DataFrame(columns=["city_pos", "edge_dist"])

    search_env = expanded_bbox(focal_geom, search_distance)

    candidate_positions = sindex_query_positions(city_sindex, search_env)

    if len(candidate_positions) == 0:
        return pd.DataFrame(columns=["city_pos", "edge_dist"])

    rows = []

    for pos in candidate_positions:

        # Remove focal building by ID if possible.
        if city_id_array is not None and focal_unique_id is not None:
            try:
                if city_id_array[pos] == focal_unique_id:
                    continue
            except Exception:
                pass

        candidate_geom = city_gdf.geometry.iloc[pos]

        if candidate_geom is None or candidate_geom.is_empty:
            continue

        # Remove focal building by exact geometry if possible.
        if remove_exact_self and focal_wkb is not None:
            try:
                if candidate_geom.wkb == focal_wkb:
                    continue
            except Exception:
                pass

        try:
            d = focal_geom.distance(candidate_geom)
        except Exception:
            continue

        if np.isfinite(d) and d <= search_distance:
            rows.append({
                "city_pos": int(pos),
                "edge_dist": float(d)
            })

    if not rows:
        return pd.DataFrame(columns=["city_pos", "edge_dist"])

    out = pd.DataFrame(rows)

    out = out.sort_values(
        ["edge_dist", "city_pos"],
        ascending=[True, True]
    ).reset_index(drop=True)

    return out


def get_edge_knn_candidates_adaptive(
    focal_geom,
    city_gdf,
    city_sindex,
    max_k,
    initial_search_distance,
    max_search_distance,
    expansion_factor=2.0,
    city_id_array=None,
    focal_unique_id=None,
    focal_wkb=None
):
    """
    Finds enough edge-distance candidates for k-nearest-neighbor variables.

    The search expands until at least max_k exact-distance candidates are found,
    or until max_search_distance is reached.
    """

    search_distance = float(initial_search_distance)
    max_search_distance = float(max_search_distance)

    best = pd.DataFrame(columns=["city_pos", "edge_dist"])

    while search_distance <= max_search_distance:

        candidates = get_edge_distance_candidates(
            focal_geom=focal_geom,
            city_gdf=city_gdf,
            city_sindex=city_sindex,
            search_distance=search_distance,
            city_id_array=city_id_array,
            focal_unique_id=focal_unique_id,
            focal_wkb=focal_wkb,
            remove_exact_self=True
        )

        if len(candidates) >= max_k:
            return candidates

        if len(candidates) > len(best):
            best = candidates

        search_distance *= expansion_factor

    # Final attempt exactly at max_search_distance in case the loop skipped over it.
    candidates = get_edge_distance_candidates(
        focal_geom=focal_geom,
        city_gdf=city_gdf,
        city_sindex=city_sindex,
        search_distance=max_search_distance,
        city_id_array=city_id_array,
        focal_unique_id=focal_unique_id,
        focal_wkb=focal_wkb,
        remove_exact_self=True
    )

    if len(candidates) > len(best):
        best = candidates

    return best


def read_existing_tessellation_for_block(block_folder, block_crs):
    """
    Read the building_level_tessellation.gpkg created in Step 3.

    Required field:
      context_bldg_id

    cell_area_m2 is calculated directly from tessellation geometry here rather
    than being expected as an attribute from Step 3.
    """

    tess_path = os.path.join(block_folder, existing_tess_gpkg_name)

    if not os.path.exists(tess_path):
        raise FileNotFoundError(f"Existing tessellation GeoPackage not found: {tess_path}")

    tess = gpd.read_file(tess_path, layer=existing_tess_layer)

    if len(tess) == 0:
        raise ValueError(f"Existing tessellation layer is empty: {tess_path}")

    if tess.crs != block_crs:
        tess = tess.to_crs(block_crs)

    if "context_bldg_id" not in tess.columns:
        raise ValueError(
            f"'context_bldg_id' field not found in existing tessellation: {tess_path}"
        )

    tess["geometry"] = tess.geometry.apply(clean_geom)
    tess = tess[tess.geometry.notna() & ~tess.geometry.is_empty].copy()

    if len(tess) == 0:
        raise ValueError(f"No valid tessellation geometries remain after cleaning: {tess_path}")

    # If duplicates exist, dissolve to one cell per context_bldg_id.
    if tess["context_bldg_id"].duplicated().any():
        print("  WARNING: duplicate context_bldg_id values in existing tessellation; dissolving.")
        tess = tess[["context_bldg_id", "geometry"]].dissolve(
            by="context_bldg_id",
            as_index=False
        )

    tess["context_bldg_id"] = pd.to_numeric(
        tess["context_bldg_id"], errors="raise"
    ).astype("int64")

    # Step 3 stores tessellation geometry only. Step 4 owns this derived measure.
    tess["cell_area_m2"] = tess.geometry.area

    tess_attrs = pd.DataFrame(
        tess[["context_bldg_id", "cell_area_m2"]].copy()
    )

    return tess, tess_attrs


def attach_existing_tessellation_attributes(block_buildings, tess_attrs):
    """
    Join tessellation cell area to source buildings by context_bldg_id and
    calculate bldg_coverage_ratio using the building layer's area field.
    """

    if "context_bldg_id" not in block_buildings.columns:
        raise ValueError("block_buildings is missing context_bldg_id.")

    if "context_bldg_id" not in tess_attrs.columns:
        raise ValueError("tess_attrs is missing context_bldg_id.")

    # Remove any pre-existing versions so this step is authoritative.
    for col in ["cell_area_m2", coverage_field]:
        if col in block_buildings.columns:
            block_buildings = block_buildings.drop(columns=[col])

    out = block_buildings.merge(
        tess_attrs[["context_bldg_id", "cell_area_m2"]],
        on="context_bldg_id",
        how="left"
    )

    out[area_field] = pd.to_numeric(out[area_field], errors="coerce")
    out["cell_area_m2"] = pd.to_numeric(out["cell_area_m2"], errors="coerce")
    out[coverage_field] = np.nan

    valid = (
        out[area_field].notna()
        & out["cell_area_m2"].notna()
        & (out["cell_area_m2"] > 0)
    )

    out.loc[valid, coverage_field] = (
        out.loc[valid, area_field] / out.loc[valid, "cell_area_m2"]
    )

    n_missing = out["cell_area_m2"].isna().sum()

    if n_missing > 0:
        print(
            f"  WARNING: {n_missing:,} building(s) did not receive a tessellation cell."
        )

    return out


def compute_edge_distance_features_for_block(
    block_gdf,
    city_gdf,
    city_sindex,
    city_area,
    city_log_area,
    neighbor_ks,
    area_summary_ks,
    radius_values,
    initial_knn_search_distance,
    max_knn_search_distance,
    knn_expansion_factor,
    city_id_array=None,
    unique_building_id_field=None
):
    """
    Computes building-context variables using polygon-to-polygon edge distance.

    Neighbor universe:
      city-wide building polygons

    Output observations:
      buildings inside the current block folder

    For radius variables:
      If a neighbor building is within the radius based on edge distance, the
      neighbor's entire building area is included.
    """

    block_gdf = block_gdf.copy()

    max_k = max(neighbor_ks)
    max_radius = max(radius_values)

    # Initialize k-nearest output fields.
    for k in neighbor_ks:
        block_gdf[f"dist_nn{k}"] = np.nan

    # Initialize k-nearest area summary fields.
    for k in area_summary_ks:
        block_gdf[f"mean_area_nn{k}"] = np.nan
        block_gdf[f"sum_area_nn{k}"] = np.nan
        block_gdf[f"mean_log_area_nn{k}"] = np.nan

    # Initialize fixed-radius fields.
    for r in radius_values:
        block_gdf[f"count_bldgs_{r}m"] = 0
        block_gdf[f"sum_area_{r}m"] = 0.0
        block_gdf[f"mean_area_{r}m"] = np.nan
        block_gdf[f"mean_log_area_{r}m"] = np.nan

    n_block = len(block_gdf)

    for counter, (idx, row) in enumerate(block_gdf.iterrows(), start=1):

        if counter % 500 == 0:
            print(f"    Edge-distance features calculated for {counter:,}/{n_block:,} buildings...")

        focal_geom = row.geometry

        if focal_geom is None or focal_geom.is_empty:
            continue

        try:
            focal_wkb = focal_geom.wkb
        except Exception:
            focal_wkb = None

        focal_unique_id = get_focal_unique_id(row, unique_building_id_field)

        # ---------------------------------------------------------------------
        # K-nearest edge-distance neighbors
        # ---------------------------------------------------------------------

        knn_candidates = get_edge_knn_candidates_adaptive(
            focal_geom=focal_geom,
            city_gdf=city_gdf,
            city_sindex=city_sindex,
            max_k=max_k,
            initial_search_distance=initial_knn_search_distance,
            max_search_distance=max_knn_search_distance,
            expansion_factor=knn_expansion_factor,
            city_id_array=city_id_array,
            focal_unique_id=focal_unique_id,
            focal_wkb=focal_wkb
        )

        if len(knn_candidates) > 0:

            dists = knn_candidates["edge_dist"].to_numpy(dtype=float)
            city_positions = knn_candidates["city_pos"].to_numpy(dtype=int)

            for k in neighbor_ks:
                if len(dists) >= k:
                    block_gdf.at[idx, f"dist_nn{k}"] = dists[k - 1]

            for k in area_summary_ks:
                if len(city_positions) >= k:
                    k_pos = city_positions[:k]

                    neighbor_areas = city_area[k_pos]
                    neighbor_log_areas = city_log_area[k_pos]

                    block_gdf.at[idx, f"mean_area_nn{k}"] = np.nanmean(neighbor_areas)
                    block_gdf.at[idx, f"sum_area_nn{k}"] = np.nansum(neighbor_areas)
                    block_gdf.at[idx, f"mean_log_area_nn{k}"] = np.nanmean(neighbor_log_areas)

        # ---------------------------------------------------------------------
        # Fixed-radius edge-distance neighbors
        # ---------------------------------------------------------------------

        radius_candidates = get_edge_distance_candidates(
            focal_geom=focal_geom,
            city_gdf=city_gdf,
            city_sindex=city_sindex,
            search_distance=max_radius,
            city_id_array=city_id_array,
            focal_unique_id=focal_unique_id,
            focal_wkb=focal_wkb,
            remove_exact_self=True
        )

        if len(radius_candidates) == 0:
            continue

        radius_dists = radius_candidates["edge_dist"].to_numpy(dtype=float)
        radius_positions = radius_candidates["city_pos"].to_numpy(dtype=int)

        for r in radius_values:

            keep_r = radius_dists <= r
            pos_r = radius_positions[keep_r]

            count_r = len(pos_r)

            block_gdf.at[idx, f"count_bldgs_{r}m"] = count_r

            if count_r > 0:
                areas_r = city_area[pos_r]
                log_areas_r = city_log_area[pos_r]

                block_gdf.at[idx, f"sum_area_{r}m"] = np.nansum(areas_r)
                block_gdf.at[idx, f"mean_area_{r}m"] = np.nanmean(areas_r)
                block_gdf.at[idx, f"mean_log_area_{r}m"] = np.nanmean(log_areas_r)

    return block_gdf


# -----------------------------------------------------------------------------
# LOAD CITY-WIDE BUILDINGS ONCE
# -----------------------------------------------------------------------------

print("=" * 80)
print("Loading city-wide buildings")
print("=" * 80)

city_gdf = gpd.read_file(city_buildings_gdb, layer=city_buildings_layer)

if area_field not in city_gdf.columns:
    raise ValueError(
        f"Required field not found in city-wide buildings: {area_field}"
    )

city_gdf = city_gdf[city_gdf.geometry.notna()].copy()
city_gdf = city_gdf[~city_gdf.geometry.is_empty].copy()

city_gdf["geometry"] = city_gdf.geometry.apply(clean_geom)
city_gdf = city_gdf[
    city_gdf.geometry.notna() & ~city_gdf.geometry.is_empty
].copy()

city_gdf[area_field] = pd.to_numeric(city_gdf[area_field], errors="coerce")
city_gdf["log_area_m2"] = safe_log_area(city_gdf[area_field])

city_area = city_gdf[area_field].to_numpy(dtype=float)
city_log_area = city_gdf["log_area_m2"].to_numpy(dtype=float)

print(f"City-wide buildings loaded: {len(city_gdf):,}")
print(f"City CRS: {city_gdf.crs}")

# Build city-wide ID array if configured.
city_id_array = make_city_id_array(city_gdf, unique_building_id_field)

if city_id_array is not None:
    print(f"Using unique ID field for self-removal: {unique_building_id_field}")
else:
    print("Using exact geometry WKB matching for self-removal.")

# Build polygon spatial index.
print("Building city-wide polygon spatial index...")
city_sindex = city_gdf.sindex
print("City-wide polygon spatial index built.")


# -----------------------------------------------------------------------------
# FIND BLOCK FOLDERS
# -----------------------------------------------------------------------------

block_folders = find_block_folders(base_folder)

print("\n" + "=" * 80)
print("Found block folders")
print("=" * 80)
print(f"Block folders found: {len(block_folders):,}")

for f in block_folders[:10]:
    print(" ", f)

if len(block_folders) > 10:
    print("  ...")

if not block_folders:
    raise FileNotFoundError(
        f"No block folders with matching .gdb files found under:\n{base_folder}"
    )

# -------------------------------------------------------------------------
# OPTIONAL TEST MODE
# -------------------------------------------------------------------------
# To test only one block folder first, uncomment this block:
#
# block_folders = [
#     r"E:\World Bank deliverbale 1\_analysis\large_pop_blocks\_1978"
# ]
#
# -------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PROCESS EACH BLOCK FOLDER
# -----------------------------------------------------------------------------

summary_rows = []

for block_folder in block_folders:

    block_id = os.path.basename(block_folder)
    gdb_path = os.path.join(block_folder, f"{block_id}.gdb")

    print("\n" + "=" * 80)
    print(f"Processing block folder: {block_id}")
    print("=" * 80)
    print(f"GDB: {gdb_path}")

    # -------------------------------------------------------------------------
    # Read block-level buildings and block boundary
    # -------------------------------------------------------------------------

    try:
        block_buildings = read_gdb_layer(gdb_path, "buildings", preserve_fid=True)
        block_buildings["context_bldg_id"] = block_buildings.index.astype("int64")
    except Exception as e:
        print("  SKIPPING: could not read buildings layer.")
        print(f"  Error: {e}")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_could_not_read_buildings",
            "error": str(e)
        })
        continue

    try:
        block_boundary = read_gdb_layer(gdb_path, "block")
    except Exception as e:
        print("  SKIPPING: could not read block layer.")
        print(f"  Error: {e}")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_could_not_read_block",
            "error": str(e)
        })
        continue

    if len(block_buildings) == 0:
        print("  SKIPPING: buildings layer is empty.")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_empty_buildings"
        })
        continue

    if len(block_boundary) == 0:
        print("  SKIPPING: block layer is empty.")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_empty_block"
        })
        continue

    if area_field not in block_buildings.columns:
        print(f"  SKIPPING: required field not found in block buildings: {area_field}")
        print("  Available fields:")
        for c in block_buildings.columns:
            print("   ", c)

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_area_field_not_found"
        })
        continue

    block_buildings = block_buildings[
        block_buildings.geometry.notna() & ~block_buildings.geometry.is_empty
    ].copy()

    block_boundary = block_boundary[
        block_boundary.geometry.notna() & ~block_boundary.geometry.is_empty
    ].copy()

    if len(block_buildings) == 0:
        print("  SKIPPING: no valid building geometries.")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_no_valid_building_geometries"
        })
        continue

    if len(block_boundary) == 0:
        print("  SKIPPING: no valid block boundary geometry.")
        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_no_valid_block_geometry"
        })
        continue

    # Match city CRS.
    if block_buildings.crs != city_gdf.crs:
        print("  Reprojecting block buildings to city CRS.")
        block_buildings = block_buildings.to_crs(city_gdf.crs)

    if block_boundary.crs != city_gdf.crs:
        print("  Reprojecting block boundary to city CRS.")
        block_boundary = block_boundary.to_crs(city_gdf.crs)

    # -------------------------------------------------------------------------
    # Own-building variables
    # -------------------------------------------------------------------------

    block_buildings["geometry"] = block_buildings.geometry.apply(clean_geom)
    block_buildings = block_buildings[
        block_buildings.geometry.notna() & ~block_buildings.geometry.is_empty
    ].copy()

    block_buildings[area_field] = pd.to_numeric(block_buildings[area_field], errors="coerce")
    block_buildings["log_area_m2"] = safe_log_area(block_buildings[area_field])

    # Retain centroids as descriptive attributes only.
    # These are NOT used for neighbor-distance calculations.
    bx, by = get_centroid_xy(block_buildings)
    block_buildings["centroid_x"] = bx
    block_buildings["centroid_y"] = by

    # context_bldg_id was preserved from the FileGDB feature ID when the
    # buildings layer was read. This matches the ID used by Step 3.
    block_buildings["block_folder"] = block_id

    print(f"  Block buildings: {len(block_buildings):,}")

    # -------------------------------------------------------------------------
    # Read existing tessellation, calculate cell area, and attach cell attributes
    # -------------------------------------------------------------------------

    try:
        print("  Reading existing building-level tessellation...")

        tess_cells, tess_attrs = read_existing_tessellation_for_block(
            block_folder=block_folder,
            block_crs=block_buildings.crs
        )

        block_buildings = attach_existing_tessellation_attributes(
            block_buildings=block_buildings,
            tess_attrs=tess_attrs
        )

        print("  Attached tessellation cell area and calculated building coverage ratio.")
        print(f"  Existing tessellation cells: {len(tess_cells):,}")
        print(f"  Mean cell_area_m2: {block_buildings['cell_area_m2'].mean():,.2f}")
        print(f"  Mean {coverage_field}: {block_buildings[coverage_field].mean():.6f}")

    except Exception as e:
        print("  SKIPPING: could not read or attach existing tessellation attributes.")
        print(f"  Error: {e}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_existing_tessellation_failed",
            "error": str(e),
            "n_buildings": len(block_buildings)
        })
        continue

    # -------------------------------------------------------------------------
    # Neighbor-context variables using polygon edge distance
    # -------------------------------------------------------------------------

    try:
        print("  Calculating polygon-edge-distance neighbor variables...")

        block_buildings = compute_edge_distance_features_for_block(
            block_gdf=block_buildings,
            city_gdf=city_gdf,
            city_sindex=city_sindex,
            city_area=city_area,
            city_log_area=city_log_area,
            neighbor_ks=neighbor_ks,
            area_summary_ks=area_summary_ks,
            radius_values=radius_values,
            initial_knn_search_distance=initial_knn_search_distance,
            max_knn_search_distance=max_knn_search_distance,
            knn_expansion_factor=knn_expansion_factor,
            city_id_array=city_id_array,
            unique_building_id_field=unique_building_id_field
        )

        print("  Finished polygon-edge-distance neighbor variables.")

    except Exception as e:
        print("  SKIPPING: edge-distance feature calculation failed.")
        print(f"  Error: {e}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "status": "skipped_edge_distance_failed",
            "error": str(e),
            "n_buildings": len(block_buildings)
        })
        continue

    # -------------------------------------------------------------------------
    # Block-level area attribute
    # -------------------------------------------------------------------------

    try:
        block_union_for_area = unary_union(block_boundary.geometry)
        block_area_m2 = block_union_for_area.area
    except Exception:
        block_area_m2 = np.nan

    block_buildings["block_area_m2"] = block_area_m2

    # -------------------------------------------------------------------------
    # Write outputs
    # -------------------------------------------------------------------------

    out_building_gpkg = os.path.join(block_folder, out_building_gpkg_name)
    out_building_csv = os.path.join(block_folder, out_building_csv_name)

    # Remove previous building-context output if overwrite is true.
    if os.path.exists(out_building_gpkg):
        if overwrite_outputs:
            try:
                os.remove(out_building_gpkg)
                print(f"  Existing output removed: {out_building_gpkg}")
            except Exception as e:
                print(f"  WARNING: could not remove existing output: {out_building_gpkg}")
                print(f"  Error: {e}")
        else:
            print(f"  Output exists and overwrite_outputs=False: {out_building_gpkg}")

    try:
        block_buildings.to_file(
            out_building_gpkg,
            layer=out_building_layer,
            driver="GPKG"
        )
        print("  Wrote building context GeoPackage:")
        print(f"  {out_building_gpkg}")

    except Exception as e:
        print("  ERROR writing building context GeoPackage.")
        print(f"  Error: {e}")

    try:
        csv_df = pd.DataFrame(block_buildings.drop(columns="geometry"))
        csv_df.to_csv(out_building_csv, index=False)
        print("  Wrote building context CSV:")
        print(f"  {out_building_csv}")

    except Exception as e:
        print("  ERROR writing building context CSV.")
        print(f"  Error: {e}")

    # -------------------------------------------------------------------------
    # Summary row
    # -------------------------------------------------------------------------

    summary_row = {
        "block_folder": block_id,
        "block_folder_path": block_folder,
        "status": "success",
        "n_buildings": len(block_buildings),
        "n_tessellation_cells_existing": len(tess_cells),
        "block_area_m2": block_area_m2,
        "mean_area_m2": block_buildings[area_field].mean(),
        "mean_log_area_m2": block_buildings["log_area_m2"].mean(),
        "mean_cell_area_m2": block_buildings["cell_area_m2"].mean(),
        "mean_bldg_coverage_ratio": block_buildings[coverage_field].mean(),
        "n_missing_cell_area_m2": block_buildings["cell_area_m2"].isna().sum(),
        "n_missing_bldg_coverage_ratio": block_buildings[coverage_field].isna().sum(),
        "out_building_gpkg": out_building_gpkg,
        "out_building_csv": out_building_csv,
        "existing_tess_gpkg": os.path.join(block_folder, existing_tess_gpkg_name)
    }

    for k in neighbor_ks:
        col = f"dist_nn{k}"
        if col in block_buildings.columns:
            summary_row[f"mean_{col}"] = block_buildings[col].mean()
            summary_row[f"median_{col}"] = block_buildings[col].median()
            summary_row[f"pct_missing_{col}"] = block_buildings[col].isna().mean() * 100

    for k in area_summary_ks:
        for prefix in ["mean_area_nn", "sum_area_nn", "mean_log_area_nn"]:
            col = f"{prefix}{k}"
            if col in block_buildings.columns:
                summary_row[f"mean_{col}"] = block_buildings[col].mean()
                summary_row[f"median_{col}"] = block_buildings[col].median()

    for r in radius_values:
        for prefix in ["count_bldgs", "sum_area", "mean_area", "mean_log_area"]:
            col = f"{prefix}_{r}m"
            if col in block_buildings.columns:
                summary_row[f"mean_{col}"] = block_buildings[col].mean()
                summary_row[f"median_{col}"] = block_buildings[col].median()

    summary_rows.append(summary_row)


# -----------------------------------------------------------------------------
# WRITE OVERALL SUMMARY
# -----------------------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)

summary_csv = os.path.join(
    base_folder,
    "building_context_features_new_dist_summary.csv"
)

summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 80)
print("Done.")
print("=" * 80)

if "status" in summary_df.columns:
    print(f"Processed successful block folders: {(summary_df['status'] == 'success').sum():,}")
else:
    print(f"Processed successful block folders: {len(summary_rows):,}")

print(f"Total summary rows: {len(summary_rows):,}")
print("Summary CSV written to:")
print(summary_csv)
